# Baseline Model — Credit Risk

Train simple models to establish a performance baseline.

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import mlflow.sklearn

# Save tracking at project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
mlflow.set_tracking_uri(f"sqlite:///{project_root}/mlflow.db")

df = pd.read_csv('../data/processed/credit_risk_clean.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (149986, 12)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberRealEstateLoansOrLines,NumberOfDependents,income_missing,total_late_payments,has_late_payment,has_severe_late
0,1,0.766127,45,0.802982,9120.0,13,6,2.0,0,2,1,0
1,0,0.957151,40,0.121876,2600.0,4,0,1.0,0,0,0,0
2,0,0.658180,38,0.085113,3042.0,2,0,0.0,0,2,1,1
3,0,0.233810,30,0.036050,3300.0,5,0,0.0,0,0,0,0
4,0,0.907239,49,0.024926,63588.0,7,1,0.0,0,1,1,0


## 1. Split Features and Target

In [2]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaled version for Logistic Regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Default rate train: {y_train.mean():.2%}')
print(f'Default rate test:  {y_test.mean():.2%}')

Train: 119988 | Test: 29998
Default rate train: 6.68%
Default rate test:  6.68%


## 2. Logistic Regression

In [3]:
mlflow.set_experiment('credit-risk')

with mlflow.start_run(run_name='logistic-regression'):
    model_lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    model_lr.fit(X_train_s, y_train)

    y_pred_lr = model_lr.predict(X_test_s)
    y_prob_lr = model_lr.predict_proba(X_test_s)[:, 1]
    auc_lr = roc_auc_score(y_test, y_prob_lr)

    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_param('scaler', 'StandardScaler')
    mlflow.log_metric('auc', auc_lr)
    mlflow.sklearn.log_model(model_lr, 'model')

    print(f'AUC: {auc_lr:.4f}')
    print()
    print(classification_report(y_test, y_pred_lr, target_names=['Paid', 'Defaulted']))

2026/04/09 22:23:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 22:23:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


AUC: 0.8620

              precision    recall  f1-score   support

        Paid       0.98      0.80      0.88     27993
   Defaulted       0.21      0.77      0.33      2005

    accuracy                           0.79     29998
   macro avg       0.60      0.78      0.61     29998
weighted avg       0.93      0.79      0.84     29998



## 3. Random Forest

In [4]:
with mlflow.start_run(run_name='random-forest'):
    model_rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    model_rf.fit(X_train, y_train)

    y_pred_rf = model_rf.predict(X_test)
    y_prob_rf = model_rf.predict_proba(X_test)[:, 1]
    auc_rf = roc_auc_score(y_test, y_prob_rf)

    mlflow.log_param('model_type', 'RandomForestClassifier')
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_metric('auc', auc_rf)
    mlflow.sklearn.log_model(model_rf, 'model')

    print(f'AUC: {auc_rf:.4f}')
    print()
    print(classification_report(y_test, y_pred_rf, target_names=['Paid', 'Defaulted']))

2026/04/09 22:23:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 22:23:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


AUC: 0.8387

              precision    recall  f1-score   support

        Paid       0.94      0.99      0.97     27993
   Defaulted       0.54      0.16      0.24      2005

    accuracy                           0.93     29998
   macro avg       0.74      0.57      0.60     29998
weighted avg       0.92      0.93      0.92     29998



## 4. Compare Results

In [5]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'AUC': [auc_lr, auc_rf],
})

print(results.to_string(index=False))
print(f'\nBest model: {results.loc[results["AUC"].idxmax(), "Model"]}')

              Model      AUC
Logistic Regression 0.861959
      Random Forest 0.838731

Best model: Logistic Regression


## 5. Register Best Model

In [6]:
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name('credit-risk')
runs = client.search_runs(experiment.experiment_id, order_by=['metrics.auc DESC'], max_results=1)
best_run = runs[0]

print(f'Best run: {best_run.info.run_name}')
print(f'AUC: {best_run.data.metrics["auc"]:.4f}')

model_uri = f'runs:/{best_run.info.run_id}/model'
result = mlflow.register_model(model_uri, 'credit-risk-model')
print(f'Registered: {result.name} v{result.version}')

Best run: mlp-pytorch
AUC: 0.8675


Registered model 'credit-risk-model' already exists. Creating a new version of this model...


MlflowException: Unable to find a logged_model with artifact_path model under run 545dd6f360c64595a5d12d9eb3229ce3